# CSC2107 Assignment 5.1: Naive vs. Memoized Fibonacci

# NAMES OF THE GROUP MEMBERS
1. Othieno david
2. Tayebwa pedson
3. Nakire jackline
4. kyemba mohammed
5. Ssemwanga robert
6. Ghai samuel mach

Run the cells top to bottom. Note: the Algorithm A experiment in section 2 takes a few minutes
(n = 42 alone is ~95 s in plain Python). To run faster, shorten the `A_SIZES` list.

In [1]:
import csv
import sys
import threading
import time

import matplotlib.pyplot as plt

%matplotlib inline

## Part 1: Implementations (with call counters)

In [2]:
calls_A = 0
calls_B = 0
fibtable = {}


def fib_A(n):
    """Algorithm A: plain recursion."""
    global calls_A
    calls_A += 1
    if n == 0 or n == 1:
        value = n
    else:
        value = fib_A(n - 1) + fib_A(n - 2)
    return value


def fib_B(n):
    """Algorithm B: recursion with memoization (dict as fibtable)."""
    global calls_B
    calls_B += 1
    if n in fibtable:
        return fibtable[n]
    if n == 0 or n == 1:
        value = n
    else:
        value = fib_B(n - 1) + fib_B(n - 2)
    fibtable[n] = value
    return value


def run_A(n):
    global calls_A
    calls_A = 0
    t0 = time.perf_counter()
    v = fib_A(n)
    return v, calls_A, time.perf_counter() - t0


def run_B(n):
    global calls_B
    calls_B = 0
    fibtable.clear()
    t0 = time.perf_counter()
    v = fib_B(n)
    return v, calls_B, time.perf_counter() - t0


def fib_iter(n):
    """Iterative reference used only to check correctness."""
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

### Correctness check (n = 0 … 20)

In [3]:
default_limit = sys.getrecursionlimit()
print("Default recursion limit:", default_limit)

for n in range(21):
    assert run_A(n)[0] == run_B(n)[0] == fib_iter(n), n
print("A and B agree with the iterative Fibonacci for n = 0..20")

Default recursion limit: 1000
A and B agree with the iterative Fibonacci for n = 0..20


## Part 2: Experiments

### Algorithm A (stops after the first run that exceeds 60 s)

In [ ]:
rows = []
A_SIZES = [5, 10, 15, 20, 25, 30, 35, 36, 37, 38, 39, 40, 41, 42]

for n in A_SIZES:
    v, c, t = run_A(n)
    assert c == 2 * fib_iter(n + 1) - 1          # calls(n) = 2F(n+1) - 1
    rows.append(("A", n, c, t))
    print(f"A n={n:<4} calls={c:<12} time={t:.4f}s", flush=True)
    if t > 60:
        break

rate = rows[-1][2] / rows[-1][3]
print(f"Measured call rate (A): {rate:,.0f} calls/s")

A n=5    calls=15           time=0.0000s
A n=10   calls=177          time=0.0001s
A n=15   calls=1973         time=0.0008s
A n=20   calls=21891        time=0.0049s
A n=25   calls=242785       time=0.0458s
A n=30   calls=2692537      time=0.4912s
A n=35   calls=29860703     time=6.7893s
A n=36   calls=48315633     time=9.6942s
A n=37   calls=78176337     time=14.3546s
A n=38   calls=126491971    time=23.9825s


### Algorithm B (table cleared before each run)

In [ ]:
sys.setrecursionlimit(max(default_limit, 3000))   # n = 1000 needs depth ~1001
for n in [10, 50, 100, 200, 500, 1000]:
    v, c, t = run_B(n)
    assert c == 2 * n - 1 and v == fib_iter(n)    # calls(n) = 2n - 1
    rows.append(("B", n, c, t))
    print(f"B n={n:<4} calls={c:<6} time={t:.6f}s", flush=True)
sys.setrecursionlimit(default_limit)

with open("results.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["algorithm", "n", "calls", "time_s"])
    for r in rows:
        w.writerow([r[0], r[1], r[2], f"{r[3]:.6f}"])

### Results table

In [ ]:
print(f"{'alg':<4}{'n':>6}{'calls':>16}{'time (s)':>14}")
for alg, n, c, t in rows:
    print(f"{alg:<4}{n:>6}{c:>16,}{t:>14.6f}")

### Graphs

A grows exponentially, so it needs a log y-axis (an exponential becomes a straight line). B is linear, so linear axes are used.

In [ ]:
A = [r for r in rows if r[0] == "A"]
B = [r for r in rows if r[0] == "B"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot([r[1] for r in A], [r[2] for r in A], "o-", label="A (naive)")
ax[0].set_yscale("log")
ax[0].set_title("Algorithm A: calls vs n (log y-axis)")
ax[0].set_xlabel("n"); ax[0].set_ylabel("calls"); ax[0].grid(True, which="both", alpha=.3)
ax[0].legend()
ax[1].plot([r[1] for r in B], [r[2] for r in B], "s-", color="tab:green", label="B (memoized)")
ax[1].set_title("Algorithm B: calls vs n (linear axes)")
ax[1].set_xlabel("n"); ax[1].set_ylabel("calls"); ax[1].grid(True, alpha=.3)
ax[1].legend()
fig.tight_layout()
fig.savefig("calls_plot.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([r[1] for r in A], [r[2] for r in A], "o-", label="A (naive)")
ax.plot([r[1] for r in B], [r[2] for r in B], "s-", label="B (memoized)")
ax.set_yscale("log"); ax.set_xscale("log")
ax.set_xlabel("n (log)"); ax.set_ylabel("calls (log)")
ax.set_title("Calls vs n, both algorithms"); ax.grid(True, which="both", alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig("calls_both.png", dpi=150)
plt.show()

## Part 3.5: Extrapolating Algorithm A to n = 1000

In [ ]:
c1000 = 2 * fib_iter(1001) - 1
secs = c1000 / rate
print(f"A at n=1000: calls = {c1000:.3e}  ({len(str(c1000))} digits)")
print(f"Estimated time: {secs:.3e} s = {secs/3.15576e7:.3e} years")

## Part 4: Pushing Algorithm B to n = 1000

### 4.1 Default recursion limit

In [ ]:
for n in (100, 500, 997, 998, 1000):
    fibtable.clear()
    try:
        run_B(n)
        print("default limit: n =", n, "OK")
    except RecursionError as e:
        print("default limit: n =", n, "RecursionError:", e)